In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
from plot_utils import bar_countprop_plot

In [ ]:
df = pd.read_csv("../data/raw/bank-additional-full.csv", sep=";")

In [ ]:
df.info()

In [ ]:
df.sample(15)

In [ ]:
month_order = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
df["month"] = pd.Categorical(df["month"], categories=month_order, ordered=True)

day_of_week_order = ["mon", "tue", "wed", "thu", "fri"]
df["day_of_week"] = pd.Categorical(df["day_of_week"], categories=day_of_week_order, ordered=True)

education_order = ["illiterate", "basic.4y", "basic.6y", "basic.9y", "high.school", "professional.course", "university.degree", "unknown"]
df["education"] = pd.Categorical(df["education"], categories=education_order, ordered=True)

In [ ]:
df["conv_yes"] = (df["y"] == "yes").astype(int)

In [ ]:
df["y"].value_counts(normalize=True)

In [ ]:
df.sort_values("duration")["conv_yes"].cumsum().reset_index(drop=True).plot()
plt.xlabel("Duração contato")
plt.ylabel("Qtde conversões - Soma cumulativa")
plt.show()

In [ ]:
cols_with_unknown = ["job", "marital", "education", "default", "housing", "loan"]

(df[cols_with_unknown] == "unknown").sum(axis=1).value_counts(normalize=True).sort_index()

## Client data

In [ ]:
bins = [0, 20, 30, 40, 50, 60, float("inf")]
labels = ["<20", "20s", "30s", "40s", "50s", "60+"]
df["age_bucketed"] = pd.cut(df["age"], bins=bins, right=False, labels=labels, ordered=True)

bar_countprop_plot(df, col_name="age_bucketed")

In [ ]:
pd.crosstab(index=df["age_bucketed"], columns=df["contact"], values=df["conv_yes"], aggfunc="sum", margins=True, normalize=False)

In [ ]:
bar_countprop_plot(df, col_name="education")

In [ ]:
pd.crosstab(index=df["education"], columns=df["contact"], values=df["conv_yes"], aggfunc="sum", margins=True, normalize=False)

In [ ]:
bar_countprop_plot(df, col_name="job")

In [ ]:
pd.crosstab(index=df["job"], columns=df["contact"], values=df["conv_yes"], aggfunc="sum", margins=True, normalize=False)

In [ ]:
bar_countprop_plot(df, col_name="marital")

In [ ]:
pd.crosstab(index=df["marital"], columns=df["contact"], values=df["conv_yes"], aggfunc="sum", margins=True, normalize=False)

In [ ]:
bar_countprop_plot(df, col_name="housing")

In [ ]:
bar_countprop_plot(df, col_name="loan")

In [ ]:
bar_countprop_plot(df, col_name="default")

In [ ]:
pd.crosstab(index=df["default"], columns=df["contact"], values=df["conv_yes"], aggfunc="sum", margins=True, normalize=False)

## Current campaign (last contact)

In [ ]:
bar_countprop_plot(df, col_name="month")

In [ ]:
pd.crosstab(index=df["month"], columns=df["contact"], values=df["conv_yes"], aggfunc="sum", margins=True, normalize=False)

In [ ]:
bar_countprop_plot(df, col_name="day_of_week")

In [ ]:
bar_countprop_plot(df, col_name="contact")

## Campaign (other)

In [ ]:
n_contacts_threshold = 10
df["campaign_bucketed"] = df["campaign"].where(df["campaign"] < n_contacts_threshold, other=n_contacts_threshold)
df["campaign_bucketed"] = pd.Categorical(df["campaign_bucketed"], ordered=True)

bar_countprop_plot(df, col_name="campaign_bucketed")

In [ ]:
n_contacts_threshold = 2
df["previous_bucketed"] = df["previous"].where(df["previous"] < n_contacts_threshold, other=n_contacts_threshold)
df["previous_bucketed"] = pd.Categorical(df["previous_bucketed"], ordered=True)

bar_countprop_plot(df, col_name="previous_bucketed")

In [ ]:
pd.crosstab(index=df["previous_bucketed"], columns=df["contact"], values=df["conv_yes"], aggfunc="sum", margins=True, normalize=False)

In [ ]:
pd.crosstab(df["pdays"] != 999, df["y"], margins=True, normalize=False)

In [ ]:
bar_countprop_plot(df, col_name="poutcome")

In [ ]:
pd.crosstab(index=df["poutcome"], columns=df["contact"], values=df["conv_yes"], aggfunc="sum", margins=True, normalize=False)

## Economic context

In [ ]:
econ_cols = ["emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"]

sns.heatmap(df[econ_cols].corr(), annot=True, cmap="coolwarm", center=0)
plt.title("Economic context - Correlation")
plt.show()

In [ ]:
bins = [0, 1, 2, 3, 4, float("inf")]
labels = ["0-1", "1-2", "2-3", "3-4", "4+"]
df["euribor3m_bucketed"] = pd.cut(df["euribor3m"], bins=bins, right=False, labels=labels, ordered=True)

bar_countprop_plot(df, col_name="euribor3m_bucketed")

In [ ]:
pd.crosstab(index=df["euribor3m_bucketed"], columns=df["contact"], values=df["conv_yes"], aggfunc="sum", margins=True, normalize=False)

In [ ]:
df["cons.conf.idx"].value_counts().sort_index()

In [ ]:
_, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10))

count_by_conf = df.groupby(by=["cons.conf.idx"]).size()
count_by_conf.sort_index().plot(kind="bar", ax=ax1)
plt.xlabel("cons.conf.idx")
plt.ylabel("Ocorrências")

conv_by_conf = df.groupby(by=["cons.conf.idx"])["conv_yes"].mean()
conv_by_conf.sort_index().plot(marker="o", ax=ax2)
plt.xlabel("cons.conf.idx")
plt.ylabel("Taxa de conversão")

plt.tight_layout()
plt.show()